In [6]:
import pandas as pd

# Leer archivo
df = pd.read_csv("KDRAMAS BASE CUATRO MIL (FORMATO BIEN).csv", sep=';', on_bad_lines='skip')

# Limpiar nombres de columnas (eliminar espacios en blanco)
df.columns = df.columns.str.strip()

# Separar las plataformas usando un patrón de regex más robusto
# Esto dividirá por dos o más espacios, comas o pipes.
plataformas = (
    df["Plataformas"]
    .dropna()
    .str.split(r'\s{2,}|[,|]', regex=True)
    .explode()
    .str.strip()
)

# Contar apariciones
conteo = (
    plataformas
    .value_counts()
    .reset_index()
)

conteo.columns = ["Plataforma", "Cantidad"]

# Filtra y elimina las filas que contienen valores nulos o nombres de columna incorrectos
conteo = conteo[~conteo['Plataforma'].isin(['Column7', 'donde-ver'])].dropna(subset=['Plataforma'])

# Ensure all platform names are stripped of whitespace for consistent matching
conteo['Plataforma'] = conteo['Plataforma'].str.strip()

# --- General cleaning: remove common descriptive suffixes ---
# This step removes ' Subscription', ' Free', ' Purchase' from platform names generically.
conteo['Plataforma'] = conteo['Plataforma'].str.replace(r' Subscription', '', regex=True)
conteo['Plataforma'] = conteo['Plataforma'].str.replace(r' Free', '', regex=True)
conteo['Plataforma'] = conteo['Plataforma'].str.replace(r' Purchase', '', regex=True)
conteo['Plataforma'] = conteo['Plataforma'].str.strip() # Re-strip after generic replacements
# -----------------------------------------------------------

# --- Robust normalization for iQIYI platforms ---
# This step ensures all variations of 'iQIYI' are mapped to 'iQIYI' before other specific mappings.
conteo.loc[conteo['Plataforma'].str.contains('iQIYI', na=False, case=False), 'Plataforma'] = 'iQIYI'
# ------------------------------------------------

# --- Robust normalization for SBS World ---
# This ensures any platform containing 'SBS' is mapped to 'SBS World'.
conteo.loc[conteo['Plataforma'].str.contains('SBS', na=False, case=False), 'Plataforma'] = 'SBS World'
# ------------------------------------------

# Define a mapping for unifying other platform names
# Some of these mappings might become redundant after generic cleaning, but are kept for robustness.
platform_mapping = {
    'Viki Subscription (sub)': 'Viki',
    'Viki Free (sub)': 'Viki',
    'Netflix Subscription (sub)': 'Netflix',
    'Netflix Subscription': 'Netflix',
    'Kocowa Subscription (sub)': 'Kocowa',
    'Kocowa Free (sub)': 'Kocowa',
    'Kocowa Subscription': 'Kocowa',
    'Roku Subscription': 'Roku',
    'Roku Free': 'Roku',
    'Roku Free (sub)': 'Roku',
    'Apple TV Subscription': 'Apple TV',
    'Apple TV Subscription (sub)': 'Apple TV',
    'Prime Video Subscription (sub)': 'Prime Video',
    'Apple TV Free': 'Apple TV',
    'Apple TV Free (sub)': 'Apple TV',
    'Prime Video Subscription': 'Prime Video',
    'Prime Video Purchase': 'Prime Video',
    'Prime Video Purchase (sub)': 'Prime Video',
    'KBS World Free (sub)': 'KBS World Free',
    'Viki Subscription': 'Viki',
    'WeTV Subscription': 'WeTV',
    'Wavve Purchase': 'Wavve', # These will now be largely covered by generic replace
    'KUKAN Free': 'KUKAN',     # These will now be largely covered by generic replace
    'TVer Free': 'TVer',       # These will now be largely covered by generic replace
    'iWant TFC Subscription': 'iWant TFC' # These will now be largely covered by generic replace
    # All SBS variations are now handled by the .loc statement above
}

# Apply the mapping to the 'Plataforma' column (already stripped earlier)
conteo['Plataforma'] = conteo['Plataforma'].replace(platform_mapping)

# Re-aggregate the counts after unifying platform names (first pass)
conteo = conteo.groupby('Plataforma')['Cantidad'].sum().reset_index()
conteo = conteo.sort_values(by='Cantidad', ascending=False).reset_index(drop=True)

# Remove '(sub)' from platform names for aesthetic purposes in the chart
conteo['Plataforma'] = conteo['Plataforma'].str.replace(r' \(sub\)', '', regex=True)

# Re-aggregate the counts AGAIN after removing '(sub)' to catch any new duplicates
conteo = conteo.groupby('Plataforma')['Cantidad'].sum().reset_index()
conteo = conteo.sort_values(by='Cantidad', ascending=False).reset_index(drop=True)

print(conteo)

# Verification of SBS platforms after unification:
print("\nVerification of SBS platforms after unification:")
print(conteo[conteo['Plataforma'].str.contains('SBS', na=False, case=False)])

            Plataforma  Cantidad
0                 Viki       964
1               Kocowa       693
2          Prime Video       640
3              Netflix       549
4             Apple TV       375
5                 Roku       321
6                iQIYI       194
7                 WeTV       193
8                TVING       187
9                 Tubi       181
10               Wavve       154
11           SBS World       152
12           KBS World       121
13                Hulu        93
14     Disney+ Hotstar        93
15             Disney+        65
16          AsianCrush        56
17              KOK TV        42
18     PlayList Global        33
19               iflix        24
20          CheezeFilm        21
21         tvN D STORY        20
22  옛드 : 옛날 드라마 [드라맛집]        18
23            Kakao TV        17
24             K-DRAMA        15
25            72sec TV        12
26          Tooniverse        12
27       Dingo K-Drama        10
28  Lululala Story Lab         9
29   KBS D

In [7]:
import altair as alt

# Usar el DataFrame completo para incluir todas las plataformas
grafico = (
    alt.Chart(conteo) # Usar el DataFrame 'conteo' completo
    .mark_bar()
    .encode(
        x=alt.X(
            "Cantidad:Q",
            title="Cantidad de K-Dramas"
        ),
        y=alt.Y(
            "Plataforma:N",
            sort="-x",
            title="Plataforma"
        ),
        tooltip=["Plataforma", "Cantidad"]
    )
    .properties(
        width=650,
        height=2000, # Aumentar la altura para acomodar todas las plataformas
        title="Cantidad de K-Dramas por plataforma"
    )
)

grafico

alt.Chart(...)

In [8]:
grafico.save('todas_las_plataformas_k_dramas.html')
print('Gráfico guardado como todas_las_plataformas_k_dramas.html')

Gráfico guardado como todas_las_plataformas_k_dramas.html


Descarga tu gráfico HTML aquí: [todas_las_plataformas_k_dramas.html](todas_las_plataformas_k_dramas.html)

In [9]:
grafico.save('todas_las_plataformas_k_dramas.html')
print('Gráfico guardado como todas_las_plataformas_k_dramas.html')

Gráfico guardado como todas_las_plataformas_k_dramas.html


In [10]:
from google.colab import files

files.download('todas_las_plataformas_k_dramas.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
total_cantidad = conteo['Cantidad'].sum()
print(f"La suma total de la 'Cantidad' en el gráfico es: {total_cantidad}")

La suma total de la 'Cantidad' en el gráfico es: 5340


In [12]:
display(conteo)

,Plataforma,Cantidad
0,Viki,964
1,Kocowa,693
2,Prime Video,640
3,Netflix,549
4,Apple TV,375
5,Roku,321
6,iQIYI,194
7,WeTV,193
8,TVING,187
9,Tubi,181


### Cómo usar esta tabla en Flourish Studio:

1.  **Copia la tabla:** Selecciona y copia las filas de la tabla de arriba (excluyendo el índice si no lo necesitas, o copiando el resultado como CSV).
2.  **Abre Flourish Studio:** Ve a [Flourish Studio](https://flourish.studio/) e inicia sesión.
3.  **Selecciona la plantilla:** Busca y selecciona la plantilla 'Bubble chart' (o la plantilla específica que mencionaste).
4.  **Sube tus datos:** En la sección 'Data' (Datos) de Flourish, puedes pegar directamente los datos que copiaste o subir un archivo CSV. Te recomiendo pegar directamente el contenido de la tabla.
5.  **Mapea las columnas:**
    *   Para el **'Label'** (etiqueta de la burbuja), selecciona la columna `Plataforma`.
    *   Para el **'Size'** (tamaño de la burbuja), selecciona la columna `Cantidad`.
    *   Puedes explorar otras opciones de Flourish para personalizar colores, categorías, etc.

¡Con esto, deberías poder crear tu gráfico de burbujas!

In [13]:
conteo.to_csv('conteo_para_flourish.csv', index=False)

El archivo `conteo_para_flourish.csv` ha sido guardado. Puedes descargarlo haciendo clic en el icono de la carpeta a la izquierda de Colab, luego navegando hasta la ruta de tu notebook y haciendo clic derecho en el archivo para descargarlo. O puedes usar el siguiente código para descargarlo directamente:

In [14]:
from google.colab import files
files.download('conteo_para_flourish.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>